# Step 12 — real training-only no-anchor health diagnostic (6/20)

本 Notebook 只检查训练动力学，不重新决定 depth，不用短期 Reward/IC 选择参数。它创建全新的 no-anchor 训练状态，但严格复用已经验证的旧 6/20 N=1/2 registry/exact Z 和 N=3…20 historical median 初始化常数。不会恢复旧模型、optimizer、scheduler、anchor 或 checkpoint。

### 第 0 格：只检查配置，不开始真实计算

下一格只打印安全开关、CUDA、F/D/E/L、旧诊断来源、新 run 路径、工作参数和 health 证据门槛，通常数秒内完成。首次运行保持 `RUN_MODE='new'`。确认无误后，把 `RUN_REAL_STEP12=False` 改为 `True`，再继续运行后面的格子。

In [ ]:
import json, sys, torch
from dataclasses import asdict
from pathlib import Path
from time import perf_counter
import pandas as pd

root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'factor_gfn').is_dir())
if str(root) not in sys.path: sys.path.insert(0, str(root))

from factor_gfn.gfn import (
    ExhaustiveRegistry,
    GFNTrainer,
    LogZInitializationHealthConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    TrainingConfig,
    build_formal_stage5_no_anchor_6_20_config,
    build_log_z_initialization_health,
    build_real_reward_data_context,
)
from factor_gfn.gfn.diagnostic_support import (
    PhaseTrackingRewardProvider,
    configure_registry_once,
    progress_heartbeat,
    run_training_with_progress,
)

RUN_REAL_STEP12 = False
RUN_MODE = 'new'  # 中断后只可显式改为 'resume'；不会读取旧 schema/run
DEVICE = 'cuda:0'
RUN_NAME = 'step12_no_anchor_health_seed42'
LOGICAL_BATCHES = 32  # 初始总预算，不代表每个 N 自动证据充分
SOURCE_DIAGNOSTIC_ROOT = root / 'runs' / 'complexity_diagnostic_6_20' / 'manual_diagnostic_6_20_seed42'
SOURCE_REGISTRY = SOURCE_DIAGNOSTIC_ROOT / 'exhaustive_registry.sqlite3'
RUN_ROOT = root / 'runs' / 'step12_no_anchor_6_20' / RUN_NAME
LATEST_CHECKPOINT = RUN_ROOT / 'latest_no_anchor.pt'

training = TrainingConfig(
    batch_size=8,
    learning_rate=1e-4,
    log_z_learning_rate=1e-2,
    max_steps=LOGICAL_BATCHES + 2,
    model_gradient_clip_norm=5.0,
    log_z_gradient_clip_norm=5.0,
    seed=42,
)
config = build_formal_stage5_no_anchor_6_20_config(training=training)
health_config = LogZInitializationHealthConfig()
strata = config.resolved_strata()
print({
    'run_enabled': RUN_REAL_STEP12,
    'run_mode': RUN_MODE,
    'device': DEVICE,
    'source_diagnostic_root': str(SOURCE_DIAGNOSTIC_ROOT),
    'run_root': str(RUN_ROOT),
    'F': strata.feasible_node_counts,
    'D': strata.discovery_node_counts,
    'E': strata.exact_normalizer_node_counts,
    'L': strata.learned_normalizer_node_counts,
    'logical_batch_budget': LOGICAL_BATCHES,
    'health_evidence_config': asdict(health_config),
    'config_fingerprint': config.fingerprint(),
}, flush=True)
print('这些是 Step 12 工程 baseline 与诊断门槛，不是最终冻结的训练超参数。', flush=True)
print('确认后把 RUN_REAL_STEP12 改成 True；若同名目录已存在，不要覆盖，先判断应使用 resume 还是另建 run。', flush=True)


### 第 1 格：建立新 run，并只读复用旧 6/20 初始化信息

确认安全开关后运行。该格会加载真实 training-only 数据、检查行业中性化与 CUDA，然后以 SQLite 真正只读模式打开旧 N=1/2 registry，重新核对当前 6/636 个 canonical hash、语义指纹和 exact Z，并导入 N=3…20 historical median 常数。

不会重新执行 642 条 RealReward，也不会恢复旧训练状态。主要耗时来自真实数据/provider 首次加载；有缓存时通常约 1–5 分钟，具体以机器和磁盘为准，期间每 20 秒输出 elapsed heartbeat。若中断，新 run 已生成 checkpoint 后可将 `RUN_MODE` 改为 `resume`，从第 0 格顺序重跑。

In [ ]:
if not RUN_REAL_STEP12: raise RuntimeError('安全停止：先检查上一格，再显式设置 RUN_REAL_STEP12=True')
if RUN_MODE not in {'new', 'resume'}: raise ValueError("RUN_MODE 只能是 'new' 或 'resume'")
if not DEVICE.startswith('cuda:') or not torch.cuda.is_available(): raise RuntimeError('Step 12 必须显式使用 CUDA，不回落 CPU')
for required in ('diagnostic_summary.json', 'diagnostic_context.json', 'diagnostic_checkpoint.pt', 'exhaustive_registry.sqlite3'):
    if not (SOURCE_DIAGNOSTIC_ROOT / required).is_file(): raise FileNotFoundError(SOURCE_DIAGNOSTIC_ROOT / required)

device = torch.device(DEVICE)
torch.cuda.set_device(device)
torch.cuda.reset_peak_memory_stats(device)
if RUN_MODE == 'new':
    RUN_ROOT.mkdir(parents=True, exist_ok=False)
elif not RUN_ROOT.is_dir():
    raise FileNotFoundError('resume 要求既有同名 Step 12 run 目录')

print('[provider] starting training-only data/context load; heartbeat every 20s', flush=True)
provider_started = perf_counter()
with progress_heartbeat('provider load', interval_seconds=20.0):
    context = build_real_reward_data_context(paths=RealRewardDataPaths())
    base_provider = RealRewardProvider(context, config.reward)
    provider = PhaseTrackingRewardProvider(base_provider, audit_path=RUN_ROOT / 'reward_phase_audit.jsonl')
manifest = provider.manifest()
assert manifest['data_scope'] == 'training_only' and manifest['validation_oos_loaded'] is False
assert manifest['industry_neutralization']['enabled'] and base_provider.reward_config.candidate_industry_neutralization
print(f'[provider] ready in {perf_counter()-provider_started:.1f}s; fingerprint={provider.fingerprint()}', flush=True)

print('[reuse] opening historical registry in SQLite read-only mode', flush=True)
registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
trainer = GFNTrainer(config, provider, device=device)
with progress_heartbeat('registry equivalence proof', interval_seconds=20.0):
    configure_registry_once(trainer, registry)

if RUN_MODE == 'resume':
    if not LATEST_CHECKPOINT.is_file(): raise FileNotFoundError('resume 未找到 latest_no_anchor.pt')
    trainer.load_checkpoint(LATEST_CHECKPOINT)
    historical = trainer.historical_log_z_initialization
    if historical is None: raise RuntimeError('resume checkpoint 缺少 historical initialization provenance')
    print(f'[resume] step={trainer.step}, optimizer_step={trainer.optimizer_step}', flush=True)
else:
    print('[initialization] verifying old summary/context/checkpoint metadata and importing median constants', flush=True)
    with progress_heartbeat('historical initialization proof', interval_seconds=20.0):
        historical = trainer.initialize_verified_historical_log_z(SOURCE_DIAGNOSTIC_ROOT)
    trainer.save_checkpoint(LATEST_CHECKPOINT)
    print(f'[initialization] checkpoint={LATEST_CHECKPOINT}', flush=True)

initial_log_z_by_N = {
    **{n: float(trainer.tb_loss.exact_tb_log_z_by_node_count[n-1]) for n in trainer.resolved_exhaustive_node_counts},
    **historical.median_log_z_by_N,
}
if set(initial_log_z_by_N) != set(trainer.resolved_feasible_node_counts): raise RuntimeError('initial logZ coverage is incomplete')
run_context = {
    'schema': 'factor_gfn.step12_no_anchor_context.v2',
    'config_fingerprint': config.fingerprint(),
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': manifest['context_fingerprint'],
    'historical_provenance_fingerprint': historical.provenance_fingerprint,
    'source_registry': str(SOURCE_REGISTRY.resolve()),
    'registry_read_only': registry.read_only,
}
context_path = RUN_ROOT / 'step12_context.json'
if RUN_MODE == 'new':
    context_path.write_text(json.dumps(run_context, ensure_ascii=False, indent=2), encoding='utf-8')
elif json.loads(context_path.read_text(encoding='utf-8')) != run_context:
    raise RuntimeError('resume 的 Step 12 context 与当前环境不一致')
print('[reuse] verified canonical hashes', {n: len(p.canonical_structural_hashes) for n, p in trainer.exhaustive_reuse_proofs_by_N.items()}, flush=True)
print('[reuse] exact logZ', {n: initial_log_z_by_N[n] for n in trainer.resolved_exhaustive_node_counts}, flush=True)
print('[reuse] historical median count', len(historical.median_log_z_by_N), 'provenance', historical.provenance_fingerprint, flush=True)
print('[reuse] no exhaustive/calibration RealReward evaluation was executed', flush=True)


### 第 2 格：快速检查初始化表

下一格不采样、不训练，通常立即完成。它把 N=1…20 的 normalizer 类型、初始 logZ、historical calibration 样本数/IQR 和来源列成表并保存，方便在正式训练前人工核对。N=1/2 应显示 `exact_fixed`，N=3…20 应显示 `historical_median`。

In [ ]:
initialization_rows = []
for node_count in trainer.resolved_feasible_node_counts:
    calibration = historical.calibration_statistics_by_N.get(node_count, {})
    initialization_rows.append({
        'N': node_count,
        'normalizer_kind': 'exact_fixed' if node_count in trainer.resolved_exhaustive_node_counts else 'learned',
        'initialization_source': 'exact_registry' if node_count in trainer.resolved_exhaustive_node_counts else 'historical_median',
        'initial_log_z': initial_log_z_by_N[node_count],
        'historical_requested': calibration.get('calibration_requested'),
        'historical_valid': calibration.get('calibration_valid'),
        'historical_iqr': calibration.get('iqr'),
    })
initialization_frame = pd.DataFrame(initialization_rows).set_index('N')
initialization_frame.to_csv(RUN_ROOT / 'normalizer_initialization_by_N.csv')
(RUN_ROOT / 'normalizer_initialization.json').write_text(json.dumps({
    'schema': historical.schema,
    'provenance_fingerprint': historical.provenance_fingerprint,
    'initial_log_z_by_N': initial_log_z_by_N,
    'restored_training_state': historical.restored_training_state,
}, ensure_ascii=False, indent=2), encoding='utf-8')
display(initialization_frame)
assert historical.restored_training_state is False
assert trainer.step == 0 and trainer.optimizer_step == 0 and not trainer.optimizer.state if RUN_MODE == 'new' else True
print('INITIALIZATION_REUSE_READY：下一格才开始真实 conditional training。', flush=True)


### 第 3 格：运行 32 个 logical batch 的 training-health diagnostic

下一格才会进行真实 conditional training。32 batch 只是初始总预算，不保证每个 N 自动证据充分。每个 batch 开始/结束都会打印计数、loss、retry、累计耗时和 ETA；单个 batch 内若暂时没有新输出，也会每 20 秒打印 heartbeat，因此可以区分“仍在计算”和“卡死”。每个 batch 后保存 `latest_no_anchor.pt`，中断后用 `RUN_MODE='resume'` 从第 0 格顺序恢复。总时间可能从几十分钟到数小时；第一个 batch 完成后会根据真实速度持续更新 ETA。

结束后会按 N 输出 valid trajectory、successful gradient exposure、initialization/pre-update、early、late TB delta，以及 initial/current/net-change logZ。暴露不足只能是 `insufficient_evidence`，不会自动触发 calibration。

In [ ]:
remaining_batches = max(0, LOGICAL_BATCHES - trainer.step)
training_path = RUN_ROOT / 'training_health.jsonl'
trajectory_path = RUN_ROOT / 'trajectory_tb_diagnostics.jsonl'
if remaining_batches:
    print(f'[training] remaining logical batches={remaining_batches}; per-batch progress and 20s heartbeat enabled', flush=True)
    with provider.phase('discovery'):
        run_training_with_progress(
            trainer,
            logical_batches=remaining_batches,
            checkpoint_path=LATEST_CHECKPOINT,
            checkpoint_every=1,
            training_audit_path=training_path,
            trajectory_audit_path=trajectory_path,
        )
else:
    print('[training] requested budget already present in resumed checkpoint', flush=True)

training_rows = [json.loads(line) for line in training_path.read_text(encoding='utf-8').splitlines() if line.strip()]
trajectory_rows = [json.loads(line) for line in trajectory_path.read_text(encoding='utf-8').splitlines() if line.strip()]
current_log_z_by_N = {
    **{n: float(trainer.tb_loss.exact_tb_log_z_by_node_count[n-1]) for n in trainer.resolved_exhaustive_node_counts},
    **{n: float(trainer.tb_loss.log_z_by_node_count[n-1]) for n in trainer.resolved_learned_node_counts},
}
health_result = build_log_z_initialization_health(
    trajectory_rows,
    training_rows,
    node_counts=trainer.resolved_feasible_node_counts,
    learned_node_counts=trainer.resolved_learned_node_counts,
    initial_log_z_by_N=initial_log_z_by_N,
    current_log_z_by_N=current_log_z_by_N,
    config=health_config,
)
health_rows = []
for node_count, item in health_result.per_N.items():
    phases = item['tb_delta_by_phase']
    health_rows.append({
        'N': node_count,
        'kind': item['normalizer_kind'],
        'status': item['status'],
        'valid_trajectories': item['valid_trajectory_count'],
        'successful_gradient_exposures': item['successful_gradient_exposure_count'],
        'initial_delta_mean': phases['initialization_pre_update']['mean'],
        'initial_delta_std': phases['initialization_pre_update']['std'],
        'early_delta_mean': phases['early']['mean'],
        'early_delta_std': phases['early']['std'],
        'late_delta_mean': phases['late']['mean'],
        'late_delta_std': phases['late']['std'],
        'initial_log_z': item['initial_log_z'],
        'current_log_z': item['current_log_z'],
        'net_change_log_z': item['net_change_log_z'],
        'directional_update_fraction': item['directional_log_z_update_fraction'],
    })
health_frame = pd.DataFrame(health_rows).set_index('N')
health_frame.to_csv(RUN_ROOT / 'log_z_initialization_health_by_N.csv')
pd.DataFrame(training_rows).to_json(RUN_ROOT / 'training_health.json', orient='records', force_ascii=False, indent=2)
pd.DataFrame(health_result.enriched_trajectory_rows).to_csv(RUN_ROOT / 'trajectory_tb_health_phases.csv', index=False)
(RUN_ROOT / 'log_z_initialization_health.json').write_text(json.dumps(asdict(health_result), ensure_ascii=False, indent=2), encoding='utf-8')
display(health_frame)
training_frame = pd.json_normalize(training_rows)
training_frame.to_csv(RUN_ROOT / 'training_health.csv', index=False)
training_health_columns = [
    'logical_batch', 'optimizer_step', 'loss', 'tb_delta_rms',
    'model_gradient_norm_before_clip', 'log_z_gradient_before_clip',
    'model_gradient_clip_coefficient', 'log_z_gradient_clip_coefficient',
    'batch_wall_seconds', 'cuda_peak_memory_bytes',
]
display(training_frame[training_health_columns])
health_counters = {
    'requested_count_by_N': dict(trainer.requested_count_by_N),
    'valid_count_by_N': dict(trainer.valid_count_by_N),
    'sampled_attempt_count_by_N': dict(trainer.sampled_attempt_count_by_N),
    'retry_exhausted_count_by_N': dict(trainer.retry_exhausted_count_by_N),
    'successful_update_count_by_N': dict(trainer.successful_update_count_by_N),
}
for node_count in trainer.resolved_feasible_node_counts:
    assert health_result.per_N[node_count]['valid_trajectory_count'] == health_counters['valid_count_by_N'][node_count]
    assert health_result.per_N[node_count]['successful_gradient_exposure_count'] == health_counters['successful_update_count_by_N'][node_count]
print('[health] targeted recalibration review:', health_result.targeted_recalibration_node_counts, flush=True)
print('[health] insufficient evidence:', health_result.insufficient_evidence_node_counts, flush=True)
print('[health] 这里只给建议，不自动重置 logZ、不自动 calibration、不创建新 run。', flush=True)


### 第 4 格：验证新 checkpoint 的确定性恢复并写最终摘要

训练与 health 表完成后运行。它先保存新 no-anchor checkpoint，再从同一状态分别执行一次 source batch 和 resume batch，逐项比较 N、trajectory、loss、logZ、scheduler 和模型参数。两个真实 batch 都可能较久，总耗时约为当时单个 batch 用时的 2 倍，因此各自每 20 秒打印 heartbeat。

通过后会写出最终 summary 并打印结果目录。该 replay 只验证恢复确定性，不进入上一个单元已经冻结的 32-batch health 统计，也不会自动采纳 targeted calibration 建议。

In [ ]:
checkpoint = RUN_ROOT / 'checkpoint_before_determinism.pt'
trainer.save_checkpoint(checkpoint)
print('[determinism] source replay batch starting (1/2); heartbeat every 20s', flush=True)
source_started = perf_counter()
with provider.phase('determinism_source'):
    with progress_heartbeat('determinism source replay', interval_seconds=20.0):
        expected = trainer.train_step()
print(f'[determinism] source replay complete in {perf_counter()-source_started:.1f}s', flush=True)
expected_diag = trainer.last_discovery_trajectory_diagnostics
expected_model = {k: v.detach().clone() for k, v in trainer.model.state_dict().items()}
expected_log_z = trainer.tb_loss.log_z_by_node_count.detach().clone()
expected_scheduler = trainer.complexity_scheduler.state_dict()

resumed = GFNTrainer(config, provider, device=device)
configure_registry_once(resumed, registry)
resumed.load_checkpoint(checkpoint)
print('[determinism] resume replay batch starting (2/2); heartbeat every 20s', flush=True)
resume_started = perf_counter()
with provider.phase('determinism_resume'):
    with progress_heartbeat('determinism resume replay', interval_seconds=20.0):
        actual = resumed.train_step()
print(f'[determinism] resume replay complete in {perf_counter()-resume_started:.1f}s; comparing exact state', flush=True)
assert actual == expected and resumed.last_discovery_trajectory_diagnostics == expected_diag
assert resumed.complexity_scheduler.state_dict() == expected_scheduler
assert torch.equal(resumed.tb_loss.log_z_by_node_count, expected_log_z)
assert all(torch.equal(resumed.model.state_dict()[k], value) for k, value in expected_model.items())

summary = {
    'schema': 'factor_gfn.step12_no_anchor_training_health.v2',
    'F': trainer.resolved_feasible_node_counts,
    'D': trainer.resolved_discovery_node_counts,
    'E': trainer.resolved_exhaustive_node_counts,
    'L': trainer.resolved_learned_node_counts,
    'historical_provenance_fingerprint': historical.provenance_fingerprint,
    'registry_read_only': registry.read_only,
    'initial_log_z_by_N': initial_log_z_by_N,
    'current_log_z_by_N_after_health_budget': current_log_z_by_N,
    'initialization_health_by_N': health_result.per_N,
    'targeted_recalibration_review': health_result.targeted_recalibration_node_counts,
    'insufficient_evidence_node_counts': health_result.insufficient_evidence_node_counts,
    **health_counters,
    'logical_batch_budget': LOGICAL_BATCHES,
    'cuda_peak_memory_bytes': int(torch.cuda.max_memory_allocated(device)),
    'checkpoint_resume_exact': True,
}
(RUN_ROOT / 'step12_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
registry.close()
print('STEP12_NO_ANCHOR_HEALTH_COMPLETE', flush=True)
print('把结果目录交给 Codex 分析：', RUN_ROOT, flush=True)
